# 4 — Scope decisions before freezing the design

Source: `scope_recon.py`. **This is the only notebook that reads the 15-minute time
series**, one household at a time. It takes about 10 minutes to run.

> ⚠️ **The outputs below are already saved — just scroll and read.**
> Do **not** press *Run*. This reads the 5 GB HEAPO dataset from a local folder
> that is not attached here, so re-running produces only `FileNotFoundError`.
> Everything you need to see is stored in the cells.

### Six questions
| | Question | Answer |
|---|---|---|
| Q1 | Is the 41-household zero-intervention group a placebo arm? | **No** |
| Q2 | Can we peer-group the full fleet on metadata? | Yes — 6 strata |
| Q3 | Does the physics show up in whole-house data? | Yes |
| Q4 | Is the night setback signature recoverable? | **Yes** |
| Q5 | Is an HDD regression an adequate baseline? | **No** |
| Q6 | Does PV contaminate the heating season? | No |


In [1]:
import numpy as np
import scope_recon as S

rng = np.random.default_rng(S.SEED)
hh      = S.load("households")
meta    = S.load("meta")
pr      = S.load("protocols")
q15ov   = S.load("15min_overview")
weather = S.load_weather()

S.say(f"environment: numpy {np.__version__}, pandas {S.pd.__version__}; "
      f"scipy / sklearn / matplotlib NOT installed -- all fits are numpy least squares.")
S.say("")

environment: numpy 2.5.1, pandas 3.0.5; scipy / sklearn / matplotlib NOT installed -- all fits are numpy least squares.



> **Note on the random seed.** `rng` is created once and consumed by Q3 before Q5
> draws its 200 controls. The cells below must run in order, or Q5 sees a different
> sample and its percentages shift by about 1 point.

In [2]:
u = S.build_base(pr, q15ov)
flags = S.intervention_flags(u)

BASE -- protocol households with >= 180 d both sides (15min overview)
rows: 89   distinct households: 89
CONFIRMED: 89 households.



## Q1 — Is the zero-intervention group a valid placebo arm?

In [3]:
S.q1(u, flags)

Q1 -- IS THE ZERO-INTERVENTION GROUP A VALID PLACEBO ARM?
INTERVENED: 48   UNCHANGED: 41   night-setback-still-active (inside INTERVENED or not): 9
  of the 9 still-active cases, 2 are in INTERVENED and 7 in UNCHANGED

--- fault flags: prevalence by group ---
                         fault  INTERVENED_n  INTERVENED_%  UNCHANGED_n  UNCHANGED_%  rate_diff_pp
        heating curve too high            28          58.3           15         36.6          21.7
        heating limit too high            17          35.4            5         12.2          23.2
 night setback active (before)            23          47.9            7         17.1          30.8
        descaling too long ago            14          29.2            9         22.0           7.2
              sizing incorrect             9          18.8            5         12.2           6.6
   pipe insulation recommended             7          14.6            3          7.3           7.3
thermostatic valve recommended             3   

## Q2 — Peer-group feasibility on the full fleet

In [4]:
peer = S.q2(hh, meta)

Q2 -- PEER-GROUP FEASIBILITY ON THE FULL FLEET
CONTRADICTS THE BRIEF: meta_data.csv covers 1358 households, not all 1408. 50 households have no metadata row at all and can never be
peer-grouped on survey variables. Percentages below are over all 1408 households,
since that is the population the detector must score.

--- completeness per metadata variable (over all 1408) ---
                                   variable  non_null  pct_of_1408
                       Survey_Building_Type      1356         96.3
          Survey_HeatPump_Installation_Type      1354         96.2
               Survey_Installation_HasDryer      1349         95.8
             Survey_Installation_HasFreezer      1349         95.8
                  Survey_Building_Residents      1348         95.7
                 Survey_Building_LivingArea      1339         95.1
Survey_HeatDistribution_System_FloorHeating      1310         93.0
    Survey_HeatDistribution_System_Radiator      1310         93.0
     Survey_Installa

## Q3 — Does the physics show up in the data?

In [5]:
hdd = S.q3(hh, q15ov, weather, rng)

Q3 -- DOES THE PHYSICS SHOW UP IN THE DATA?
stratified sample: 30 households  {'air-source': np.int64(17), 'ground-source': np.int64(13)}

--- whole-household daily kWh ~ HDD (SIA) ---
  households fitted: 30  (usable R2: 29; 1 had too few days or no HDD variation)
  R2: mean=0.631  median=0.714  min=0.011  max=0.909
  quartiles: q25=0.509  q50=0.714  q75=0.811
  slope: median=1.510 kWh/HDD
  R2 < 0.3: 3 of 29 (10.3%)
  R2 distribution by 0.1 bin:
    [0.0, 0.1):   2  ##
    [0.2, 0.3):   1  #
    [0.3, 0.4):   1  #
    [0.4, 0.5):   3  ###
    [0.5, 0.6):   5  #####
    [0.6, 0.7):   2  ##
    [0.7, 0.8):   5  #####
    [0.8, 0.9):   9  #########
    [0.9, 1.0):   1  #

--- sub-metered households (n=93 at 15min): HP-only vs whole-household ---
  households fitted: 93
  R2 whole-household: mean=0.555  median=0.642
  R2 heat-pump-only : mean=0.741  median=0.826
  R2 gain from sub-metering: mean=+0.184  median=+0.110  max=+0.878
  heat pump share of total consumption: median=61.2%
  whol

## Q4 — Night setback signature

The falsification test for the whole case study. If the 21 deactivations do not
separate from the 9 still-active controls, the effect is not recoverable.

In [6]:
profiles = S.q4(u, flags, hh, weather)

Q4 -- NIGHT SETBACK SIGNATURE (the falsification test)
Profiles use LOCAL time (Europe/Zurich). The raw timestamps are UTC; without conversion the
overnight window would be shifted 1-2 h and the setback smeared into the wrong hours.
Heating-season days only (daily mean T < 12.0 C), 180 d each side of the visit.

DEACTIVATED (21): 19 of 21 households produced a usable profile
STILL ACTIVE (9): 8 of 9 households produced a usable profile
UNCHANGED (41): 39 of 41 households produced a usable profile

--- mean hourly load profile, kWh per hour (heating-season days) ---
hour |  DEACTI before   after   delta |   STILL before   after   delta |  UNCHAN before   after   delta |
---------------------------------------------------------------------------------------------------------
   0 |         1.452   1.735  +0.283 |         0.941   1.258  +0.317 |         1.770   1.846  +0.075 |
   1 |         1.407   1.641  +0.235 |         1.073   1.577  +0.504 |         1.732   1.889  +0.158 |
   2 |    

## Q5 — Weather normalisation baseline

In [7]:
q5fit = S.q5(u, hh, weather, rng)

Q5 -- WEATHER NORMALISATION BASELINE (HDD regression)
cohort: 89 protocol households + 200 random controls
For controls there is no visit, so the series is split at the MIDPOINT of its own date
range -- a pseudo-visit. This measures baseline drift with no intervention, which is the
correct null for the treated fits.

households fitted: 285  ({'control': np.int64(196), 'protocol': np.int64(89)})

CV(RMSE) monthly %     n=282  mean=  26.21  median=  17.88  q25=  12.56  q75=  28.17  min=   3.86  max= 163.69
NMBE monthly %         n=282  mean=   0.37  median=  -1.13  q25=  -9.61  q75=   8.29  min=-140.79  max= 100.00
CV(RMSE) daily %       n=283  mean=  40.67  median=  32.27  q25=  24.48  q75=  46.59  min=   8.56  max= 169.83
NMBE daily %           n=283  mean=   0.30  median=  -1.15  q25=  -9.76  q75=   8.28  min=-140.79  max= 100.00

--- ASHRAE Guideline 14 pass rates ---
  monthly  thresholds NMBE <= 5.0%, CV(RMSE) <= 15.0%
    pass CV(RMSE): 110/285 (38.6%)
    pass NMBE    :  87/285 (

## Q6 — PV contamination

In [8]:
S.q6(u, hh, weather)

Q6 -- PV CONTAMINATION
CONTRADICTS THE BRIEF: the brief states 34.65% of households have PV. In households.csv,
Installation_HasPVSystem is True for 516 of 1408 (36.65%), False for 134, and
NULL for 758 (53.8%). The real problem is not the
rate, it is that PV status is UNKNOWN for more than half the fleet. Among households where
it is known, 79.4% have PV.

--- within the 89 base households ---
  PV = True : 15
  PV = False: 72
  PV = null : 2

--- summer (JJA) vs winter (DJF) daily consumption by PV status ---
          n  summer_kWh  winter_kWh  ratio  summer_min  zero_days
pv                                                               
PV       15       12.37       45.99   3.73        3.00       0.13
no PV    72       15.76       45.01   2.92        5.37       5.43
unknown   2       12.50       27.38   2.22        0.00     129.50

  summer daily kWh: PV=12.37 vs no PV=15.76  (std diff -0.36)
  winter daily kWh: PV=45.99 vs no PV=45.01  (std diff +0.05)
  winter/summer ratio: PV=3.

## What this establishes

- **Q1 is the blocking finding.** 30 of the 41 "unchanged" households had faults
  *found* at inspection and not fixed. They are not a placebo arm. The real control
  group is the **9** still-active night-setback cases.
- **Q4 works.** The deactivated cohort shifts +4.38 pp of daily consumption into
  00:00–04:59 versus +0.82 pp for still-active controls, and the hourly profile has
  the right *shape* — overnight load up, morning recovery peak gone.
  (The audit shows the honest t-statistic is **3.2**, not the 2.8 printed here; see
  `outputs/NUMBER_AUDIT.md`.)
- **Q5 fails.** Only ~25% of households meet ASHRAE Guideline 14 with a plain HDD
  regression. A richer baseline model is required.
